## IMPORTAR LAS LIBRERIAS

Actualizar a las que se usen finalmente en tu proyecto.

In [29]:
import numpy as np
import pandas as pd
import cloudpickle

#Automcompletar rápido
%config IPCompleter.greedy=True

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

## CARGAR LOS DATOS

### Ruta del proyecto

In [2]:
ruta_proyecto = 'C:/Users/sergi/Desktop/DATA/CURSO DS4B/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/01_LEADSCORING'

### Nombre del fichero de datos

In [3]:
nombre_fichero_datos = 'Leads.csv'

### Cargar los datos

In [4]:
ruta_completa = ruta_proyecto + '/02_Datos/01_Originales/' + nombre_fichero_datos

df =  pd.read_csv(ruta_completa,index_col= 'id', sep =';')

### Seleccionar solo las variables finales

#### Cargar la lista de variables finales

In [6]:
nombre_variables_finales = ruta_proyecto + '/05_Resultados/' + 'variables_finales.pickle'

pd.read_pickle(nombre_variables_finales).sort_index().index.to_list()

['ambito_Marketing Management',
 'ambito_Select',
 'descarga_lm_No',
 'fuente_Google',
 'ocupacion_Unemployed',
 'ocupacion_Working Professional',
 'paginas_vistas_visita_mms',
 'score_actividad_mms',
 'score_perfil_mms',
 'tiempo_en_site_total_mms',
 'ult_actividad_Chat Conversation',
 'ult_actividad_Converted to Lead',
 'ult_actividad_Page Visited on Website',
 'ult_actividad_SMS Sent']

#### Apuntar (manualmente) la lista de variables finales sin extensiones

In [13]:
variables_finales = ['ambito',
                   'descarga_lm',
                   'ocupacion',
                   'paginas_vistas_visita','score_actividad','score_perfil','tiempo_en_site_total','ult_actividad']

#### Crear la matriz de variables procesos (excel)

Ir a la plantilla de Excel "Fase Producción Plantilla Procesos" y crear la matriz de variables por procesos.

#### Actualizar las importaciones

Ir arriba a la celda de importacion de paquetes y actualizarlos con los que finalmente vamos a usar.

## ESTRUCTURA DE LOS DATASETS

### Eliminar registros

#### Por duplicados

In [14]:
df.drop_duplicates(inplace = True)

C:\Users\sergi\AppData\Local\Temp\ipykernel_16744\3424306917.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace = True)


#### Por EDA

In [11]:
df = df.loc[(df.no_llamar != 'OTROS') & (df.no_enviar_email != 'Yes') & (df.ult_actividad != 'Email Bounced')]

#### Para x

Quedarse solo con las de la lista.

In [15]:
x = df[variables_finales].copy()

#### Para y

Especificar la target.

In [17]:
target = 'compra'

Crear el y.

In [18]:
y = df[target].copy()

## CREAR EL PIPELINE

### Instanciar calidad de datos

In [21]:
x.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6840 entries, 660737 to 579533
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ambito                 6231 non-null   object 
 1   descarga_lm            6840 non-null   object 
 2   ocupacion              5148 non-null   object 
 3   paginas_vistas_visita  6730 non-null   float64
 4   score_actividad        3872 non-null   float64
 5   score_perfil           3872 non-null   float64
 6   tiempo_en_site_total   6840 non-null   int64  
 7   ult_actividad          6750 non-null   object 
dtypes: float64(3), int64(1), object(4)
memory usage: 480.9+ KB


#### Crear la función

In [23]:
def calidad_datos(df):
    temp = df
    
    def imputar_moda(variable):
        return(variable.fillna(variable.mode()[0]))
    
    var_imputar_moda = ['ocupacion','ambito']
    
    temp[var_imputar_moda] = temp[var_imputar_moda].apply(imputar_moda)
    
    var_imputar_valor = ['descarga_lm','ult_actividad']
    
    valor = 'DESCONOCIDO'
    
    temp[var_imputar_valor] = temp[var_imputar_valor].fillna(valor)
    
    var_imputar_mediana = ['paginas_vistas_visita','score_actividad','score_perfil','tiempo_en_site_total']
    
    def imputar_mediana(variable):
        if pd.api.types.is_integer_dtype(variable):
            return(variable.fillna(int(variable.median())))
        else:
            return(variable.fillna(variable.median()))
    
    temp[var_imputar_mediana] = temp[var_imputar_mediana].apply(imputar_mediana)
    
    def agrupar_cat_raras(variable, criterio = 0.02):
        #Calcula las frecuencias
        frecuencias = variable.value_counts(normalize=True)
        #Identifica las que están por debajo del criterio
        temp = [cada for cada in frecuencias.loc[frecuencias < criterio].index.values]
        #Las recodifica en 'OTROS'
        temp2 = np.where(variable.isin(temp),'OTROS',variable)
        #Devuelve el resultado
        return(temp2)
    
    var_agrupar_cat_raras = ['ocupacion','ambito','descarga_lm','ult_actividad']
    
    for variable in var_agrupar_cat_raras:
        temp[variable] = agrupar_cat_raras(temp[variable],criterio = 0.02)
    
    temp['paginas_vistas_visita'] = temp['paginas_vistas_visita'].clip(0, 20)
    
    return(temp)

#### Convertirla en transformer

In [28]:
hacer_calidad_datos = FunctionTransformer(calidad_datos)

### Instanciar transformación de variables

In [30]:
var_ohe = ['ocupacion','ambito','descarga_lm','ult_actividad']
ohe = OneHotEncoder(sparse = False, handle_unknown='ignore')

var_mms = ['paginas_vistas_visita','score_actividad','score_perfil','tiempo_en_site_total']
mms = MinMaxScaler()

### Crear el pipe del preprocesamiento

#### Crear el column transformer

In [31]:
ct = make_column_transformer(
    (ohe, var_ohe),
    (mms, var_mms),
    remainder='drop')

#### Crear el pipeline del preprocesamiento

In [32]:
pipe_prepro = make_pipeline(hacer_calidad_datos, 
                            ct)

### Instanciar el modelo

#### Instanciar el algoritmo

In [33]:
modelo = LogisticRegression(n_jobs = -1, 
                       solver = 'saga',
                       penalty = 'none',
                       C = 1)

#### Crear el pipe final de entrenamiento

In [34]:
pipe_entrenamiento = make_pipeline(pipe_prepro,modelo)

#### Guardar el pipe final de entrenamiento

In [35]:
nombre_pipe_entrenamiento = 'pipe_entrenamiento.pickle'

ruta_pipe_entrenamiento = ruta_proyecto + '/04_Modelos/' + nombre_pipe_entrenamiento

with open(ruta_pipe_entrenamiento, mode='wb') as file:
   cloudpickle.dump(pipe_entrenamiento, file)

#### Entrenar el pipe final de ejecución

In [36]:
pipe_ejecucion = pipe_entrenamiento.fit(x,y)

C:\Users\sergi\miniconda3\envs\proyecto1\Lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\sergi\miniconda3\envs\proyecto1\Lib\site-packages\sklearn\linear_model\_logistic.py:1183: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(


## GUARDAR EL PIPE

### Nombre del pipe final de ejecución

In [37]:
nombre_pipe_ejecucion = 'pipe_ejecucion.pickle'

### Guardar el pipe final de ejecución

In [38]:
ruta_pipe_ejecucion = ruta_proyecto + '/04_Modelos/' + nombre_pipe_ejecucion

with open(ruta_pipe_ejecucion, mode='wb') as file:
   cloudpickle.dump(pipe_ejecucion, file)